<a href="https://colab.research.google.com/github/fwilhelmy/QB-PIER/blob/main/src/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab (Skip locally)

In [ ]:
#@title Mount your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
%matplotlib inline
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/gdrive')


## Link source code

In [ ]:
#@title Link via google drive
#@markdown Enter the path to the assignment folder in your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
import sys

folder = "/content/gdrive/MyDrive/IFT6135/HW2_2025/src" #@param {type:"string"}
!ln -Ts "$folder" /content/src 2> /dev/null

# Add the assignment folder to Python path
if '/content/src' not in sys.path:
  sys.path.insert(0, '/content/src')

In [ ]:
#@title Link via zip upload
from google.colab import files
import sys
import os
import shutil

# 1️⃣ Upload your ZIP containing only your Python code (it may contain a top-level 'src' folder)
uploaded = files.upload()  # choose your zip
if not uploaded:
    raise FileNotFoundError("No ZIP file uploaded.")
zip_name = next(iter(uploaded.keys()))
print(f"✔ Uploaded: {zip_name}")

# 2️⃣ Prepare /content/src
base_dir = "/content/src"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# 3️⃣ Extract into /content/src
!unzip -q "{zip_name}" -d "{base_dir}"
print("✔ Extracted into /content/src/")

# 4️⃣ If there's a single top-level folder inside (e.g. /content/src/src/), flatten it
entries = os.listdir(base_dir)
if len(entries) == 1 and os.path.isdir(os.path.join(base_dir, entries[0])):
    nested = os.path.join(base_dir, entries[0])
    for fname in os.listdir(nested):
        shutil.move(os.path.join(nested, fname), base_dir)
    shutil.rmtree(nested)
    print(f"✔ Flattened nested folder `{entries[0]}` into /content/src/")

# 5️⃣ Add /content/src to Python path for top-level imports
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)
    print("✔ Added '/content/src' to sys.path")


## Setting up virtual environment

In [ ]:
#@title First time setup
#@markdown Before running this cell, make sure you have mounted your Google Drive and linked the source code.
confirm_setup = False #@param {type:"boolean"}

if not confirm_setup:
  print("Please check the box to confirm you are ready to run the first time setup.")
else:
  !pip install virtualenv
  !virtualenv /content/gdrive/MyDrive/qaml_env
  !source /content/drive/MyDrive/qaml_env/bin/activate; pip install -r requirements.txt

In [ ]:
#@title Load virtual environment
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
!source /content/gdrive/MyDrive/qaml_env/bin/activate

## Checks

In [ ]:
import warnings
try:
    import torch
    if not torch.cuda.is_available():
        warnings.warn('CUDA is not available.')
    else:
        print(f'Using device: {torch.cuda.get_device_name(0)}')
except ImportError:
    pass

# Imports & Setup

In [ ]:
%matplotlib inline

from models.hybrid import (
    AmpVQC,
)

# Experiments Loading

In [ ]:
model_name = "HQNN"  #@param {type:"string"}
model_cls = AmpVQC  #@param {type:"string"}
#path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@14-35-01\\{model_name}\\{model_name}_1\\0"
path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@19-00-06\\{model_name}\\{model_name}_1\\0\\final.ckpt"
path = "C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-30@14-43-55\\AmpVQC\\AmpVQC_1\\0\\final.ckpt"

# load the model
model = model_cls.load_from_checkpoint(path, map_location='cpu')

# Experiments Training

## Experiment configuration

In [ ]:
local_config = ["device=local", "training=mid" "DATA_DIR=../data/processed", "LOGS_DIR=../logs"]

# add local configuration if running locally
colab_config = ["device=cuda", "training=mid"]


models_config = {
    "resnet18": ["model=resnet18"],
    "convnet": ["model=convnet"],
    "lenet": ["model=lenet"],
    "mlp": ["model=mlp"],
    "mlpmixer": ["model=mlpmixer"],
}

## Models Training

In [ ]:
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra
from main import train_model
from metrics import LoggingService
from datetime import datetime
import os

timestring = datetime.now().strftime("%Y-%m-%d@%H-%M-%S")
for model_name, model_config in models_config.items():
    if GlobalHydra.instance().is_initialized() : GlobalHydra.instance().clear()
    initialize(config_path="../conf", version_base=None)
    overrides = model_config + local_config
    cfg = compose(config_name="config", overrides=overrides)
    root_dir = os.path.join(cfg["LOGS_DIR"], timestring)
    LoggingService(root_dir=root_dir)
    print(f"\n>>> Run with overrides: {model_config}")
    train_model(cfg, model_name)
LoggingService().save_experiment()

# Experiment Attacks

In [ ]:
# Run transfer attacks
# attacks_metrics = run_attacks(trained_models, args)
# plot_transfer_attack_effects(attacks_metrics,
#                             title_prefix="transfer")
# from pprint import pprint
# pprint(trained_models['HQNN_Parallel_1'][0].all_metrics, depth=2)
# metrics = trained_models['HQNN_Parallel_1'][0].all_metrics

## Final

In [ ]:
def make_archive(dir_to_archive, zipped_file_name):
    try:
        shutil.make_archive(zipped_file_name, 'zip', dir_to_archive)
        print(f"Successfully created archive: {zipped_file_name}.zip")
    except Exception as e:
        print(f"Error creating archive: {e}")

In [ ]:
make_archive('logs', 'logs')
files.download('logs.zip')